# Reto 1 — Exploración Inicial de los Datos

**Máster NTIC — Bases de Datos NoSQL**  
**Dataset:** Actividad económica de locales y terrazas en Madrid (diciembre 2023) + Airbnb Madrid  

---

## Objetivos

1. **Revisión inicial** — Examinar la estructura y formato de cada fichero JSON.
2. **Revisión de campos** — Identificar tipos de datos, valores nulos, duplicados e inconsistencias.
3. **Relaciones entre ficheros** — Detectar campos comunes que permitan relacionar los datasets.

> **Nota:** Se utiliza `DuckDB` como motor de análisis en lugar de `pandas` puro, ya que permite ejecutar SQL directamente sobre ficheros JSON sin necesidad de cargarlos completamente en memoria, lo que resulta especialmente ventajoso dado el volumen de los datos (~235 MB en bruto).

## 1. Instalación e importación de librerías

In [ ]:
# Instalar dependencias (solo si es necesario)
# !pip install duckdb pandas

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print(f"DuckDB version: {duckdb.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Configuración de rutas

In [ ]:
data_raw_dir = "../data/raw"

ficheros = {
    "locales":      f"{data_raw_dir}/locales202312.json",
    "actividades":  f"{data_raw_dir}/actividadeconomica202312.json",
    "licencias":    f"{data_raw_dir}/licencias202312.json",
    "terrazas":     f"{data_raw_dir}/terrazas202312.json",
    "listings":     f"{data_raw_dir}/airbnb_listings.json",
}

con = duckdb.connect()

## 3. Revisión inicial — Volumetría y estructura

### 3.1 Número de registros por fichero

In [ ]:
volumetria = []
for nombre, ruta in ficheros.items():
    count = con.execute(f"SELECT COUNT(1) FROM read_json_auto('{ruta}')").fetchone()[0] # type: ignore
    volumetria.append({"fichero": nombre, "registros": count})

df_vol = pd.DataFrame(volumetria)
display(df_vol)

### 3.2 Esquema de cada fichero (campos y tipos)

In [ ]:
for nombre, ruta in ficheros.items():
    print(f"\n{'='*60}")
    print(f"  FICHERO: {nombre.upper()}")
    print(f"{'='*60}")
    schema_df = con.execute(f"""
        DESCRIBE SELECT * FROM read_json_auto('{ruta}') LIMIT 1
    """).fetchdf()
    print(schema_df[['column_name', 'column_type']].to_string(index=False))

### 3.3 Muestra de documentos (5 registros por fichero)

In [ ]:
for nombre, ruta in ficheros.items():
    print(f"\n{'='*60}")
    print(f"  MUESTRA: {nombre.upper()} (5 registros)")
    print(f"{'='*60}")
    muestra = con.execute(f"SELECT * FROM read_json_auto('{ruta}') LIMIT 5").fetchdf()
    display(muestra)

## 4. Revisión de campos — Calidad de datos

### 4.1 Valores nulos por campo

In [ ]:
def analizar_nulos(con, nombre, ruta, limite_cols=15):
    """Calcula el porcentaje de nulos para cada columna de un fichero JSON."""
    cols_df = con.execute(f"""
        DESCRIBE SELECT * FROM read_json_auto('{ruta}') LIMIT 1
    """).fetchdf()
    
    columnas = cols_df['column_name'].tolist()[:limite_cols]  # primeras N cols
    total = con.execute(f"SELECT COUNT(1) FROM read_json_auto('{ruta}')").fetchone()[0]
    
    nulos = []
    for col in columnas:
        nulos_count = con.execute(f"""
            SELECT COUNT(1)
            FROM read_json_auto('{ruta}')
            WHERE "{col}" IS NULL OR TRIM(CAST("{col}" AS VARCHAR)) = ''
        """).fetchone()[0]
        pct = round(nulos_count / total * 100, 2) if total > 0 else 0
        nulos.append({"campo": col, "nulos": nulos_count, "pct_nulos": pct})
    
    df_nulos = pd.DataFrame(nulos).sort_values('pct_nulos', ascending=False)
    print(f"\n{'='*60}")
    print(f"  NULOS: {nombre.upper()} (primeras {limite_cols} columnas)")
    print(f"{'='*60}")
    display(df_nulos[df_nulos['pct_nulos'] > 0].head(15))


for nombre, ruta in ficheros.items():
    analizar_nulos(con, nombre, ruta)

### 4.2 Duplicados por clave primaria

In [ ]:
claves_pk = {
    "locales":     ("locales202312.json",           "id_local"),
    "actividades": ("actividadeconomica202312.json", "id_local, id_epigrafe"),
    "licencias":   ("licencias202312.json",          "id_local, ref_licencia"),
    "terrazas":    ("terrazas202312.json",           "id_terraza"),
    "listings":    ("airbnb_listings.json",          "id"),
}

print(f"{'Fichero':<15} {'Clave Primaria':<30} {'Duplicados':>10}")
print("-" * 60)

for nombre, (fichero, pk) in claves_pk.items():
    ruta = f"{data_raw_dir}/{fichero}"
    duplicados = con.execute(f"""
        SELECT COUNT(1) as dups
        FROM (
            SELECT {pk}, COUNT(1) as cnt
            FROM read_json_auto('{ruta}')
            GROUP BY {pk}
            HAVING COUNT(1) > 1
        )
    """).fetchone()[0] # type: ignore
    print(f"{nombre:<15} {pk:<30} {duplicados:>10}")

### 4.3 Inspección de campos clave en locales

In [ ]:
# Distribución por distrito
print("--- Distribución de locales por distrito ---")
df_distritos = con.execute(f"""
    SELECT 
        desc_distrito_local AS distrito,
        COUNT(1) AS total_locales
    FROM read_json_auto('{ficheros["locales"]}')
    WHERE desc_distrito_local IS NOT NULL
    GROUP BY desc_distrito_local
    ORDER BY total_locales DESC
    LIMIT 10
""").fetchdf()
display(df_distritos)

In [ ]:
# Rango de horarios de apertura
print("--- Valores de hora_apertura1 (muestra) ---")
df_horarios = con.execute(f"""
    SELECT 
        hora_apertura1,
        COUNT(1) AS frecuencia
    FROM read_json_auto('{ficheros["locales"]}')
    WHERE hora_apertura1 IS NOT NULL AND TRIM(hora_apertura1) != ''
    GROUP BY hora_apertura1
    ORDER BY frecuencia DESC
    LIMIT 10
""").fetchdf()
display(df_horarios)

### 4.4 Inspección de campos clave en licencias

In [ ]:
# Tipos de situación de licencia
print("--- Tipos de situación de licencia ---")
df_lic = con.execute(f"""
    SELECT 
        desc_tipo_situacion_licencia AS situacion,
        COUNT(1) AS cantidad
    FROM read_json_auto('{ficheros["licencias"]}')
    GROUP BY desc_tipo_situacion_licencia
    ORDER BY cantidad DESC
""").fetchdf()
display(df_lic)

In [ ]:
# Tipos de licencia
print("--- Tipos de licencia ---")
df_tipos_lic = con.execute(f"""
    SELECT 
        desc_tipo_licencia AS tipo_licencia,
        COUNT(1) AS cantidad
    FROM read_json_auto('{ficheros["licencias"]}')
    WHERE desc_tipo_licencia IS NOT NULL
    GROUP BY desc_tipo_licencia
    ORDER BY cantidad DESC
""").fetchdf()
display(df_tipos_lic)

### 4.5 Inspección de campos clave en actividades económicas

In [ ]:
# Secciones de actividad económica
print("--- Secciones de actividad económica ---")
df_secciones = con.execute(f"""
    SELECT 
        id_seccion,
        desc_seccion,
        COUNT(1) AS total
    FROM read_json_auto('{ficheros["actividades"]}')
    WHERE id_seccion IS NOT NULL AND id_seccion != '-1'
    GROUP BY id_seccion, desc_seccion
    ORDER BY total DESC
    LIMIT 10
""").fetchdf()
display(df_secciones)

### 4.6 Inspección del dataset Airbnb (listings)

In [ ]:
# Distribución por neighbourhood (equivalente a distrito)
print("--- Alojamientos Airbnb por neighbourhood_group_cleansed ---")
df_nbhd = con.execute(f"""
    SELECT 
        neighbourhood_group_cleansed AS distrito,
        COUNT(1) AS total_alojamientos,
        ROUND(AVG(price), 2) AS precio_medio
    FROM read_json_auto('{ficheros["listings"]}')
    WHERE neighbourhood_group_cleansed IS NOT NULL
    GROUP BY neighbourhood_group_cleansed
    ORDER BY total_alojamientos DESC
""").fetchdf()
display(df_nbhd)

In [ ]:
# Distribución por tipo de habitación
print("--- Tipos de alojamiento ---")
df_room = con.execute(f"""
    SELECT 
        room_type,
        COUNT(1) AS cantidad,
        ROUND(AVG(price), 2) AS precio_medio,
        ROUND(AVG(number_of_reviews), 1) AS reseñas_media
    FROM read_json_auto('{ficheros["listings"]}')
    GROUP BY room_type
    ORDER BY cantidad DESC
""").fetchdf()
display(df_room)

In [ ]:
# Alojamientos con bedrooms = 0 (sin dormitorios)
print("--- Alojamientos sin dormitorios (bedrooms = 0 o null) ---")
df_sin_dorm = con.execute(f"""
    SELECT 
        neighbourhood_group_cleansed AS distrito,
        COUNT(1) AS sin_dormitorios
    FROM read_json_auto('{ficheros["listings"]}')
    WHERE bedrooms IS NULL OR bedrooms = 0
    GROUP BY neighbourhood_group_cleansed
    ORDER BY sin_dormitorios DESC
""").fetchdf()
display(df_sin_dorm)

## 5. Relaciones entre ficheros

### 5.1 Clave de relación principal: `id_local`

Los ficheros de locales, actividades, licencias y terrazas comparten el campo `id_local`, que actúa como clave primaria/foránea.

In [ ]:
# Verificar integridad referencial: locales sin actividad, sin licencia, sin terraza
print("--- Integridad referencial: id_local ---")

total_locales = con.execute(f"""
    SELECT COUNT(DISTINCT id_local) FROM read_json_auto('{ficheros["locales"]}')
    WHERE id_local IS NOT NULL AND id_local != 0
""").fetchone()[0] # type: ignore

locales_con_actividad = con.execute(f"""
    SELECT COUNT(DISTINCT a.id_local)
    FROM read_json_auto('{ficheros["actividades"]}') a
    WHERE EXISTS (
        SELECT 1 FROM read_json_auto('{ficheros["locales"]}') l
        WHERE l.id_local = a.id_local
    )
""").fetchone()[0] # type: ignore

locales_con_licencia = con.execute(f"""
    SELECT COUNT(DISTINCT li.id_local)
    FROM read_json_auto('{ficheros["licencias"]}') li
    WHERE EXISTS (
        SELECT 1 FROM read_json_auto('{ficheros["locales"]}') l
        WHERE l.id_local = li.id_local
    )
""").fetchone()[0] # type: ignore

locales_con_terraza = con.execute(f"""
    SELECT COUNT(DISTINCT t.id_local)
    FROM read_json_auto('{ficheros["terrazas"]}') t
    WHERE EXISTS (
        SELECT 1 FROM read_json_auto('{ficheros["locales"]}') l
        WHERE l.id_local = t.id_local
    )
""").fetchone()[0] # type: ignore

resumen_rel = pd.DataFrame([
    {"relacion": "Locales totales (PK válida)",       "id_locales_únicos": total_locales},
    {"relacion": "Locales con actividad económica",    "id_locales_únicos": locales_con_actividad},
    {"relacion": "Locales con licencia",               "id_locales_únicos": locales_con_licencia},
    {"relacion": "Locales con terraza",                "id_locales_únicos": locales_con_terraza},
])
display(resumen_rel)

### 5.2 Relación entre locales y Airbnb: campo geográfico

El fichero `airbnb_listings.json` **no contiene `id_local`**, por lo que la relación directa por clave no es posible.  
Sin embargo, ambos datasets contienen información geográfica y de distrito/barrio:

| Dataset | Campo de distrito | Campo de barrio | Coordenadas |
|---------|------------------|-----------------|-------------|
| locales | `desc_distrito_local` | `desc_barrio_local` | UTM EPSG:25830 → WGS84 |
| listings | `neighbourhood_group_cleansed` | — | GeoJSON (lon, lat) |

La estrategia adoptada es **relacionarlos por distrito** (campo geográfico común) y complementarla con **proximidad espacial** para el modelo de grafo.

In [ ]:
# Verificar solapamiento de distritos entre locales y listings
print("--- Distritos en locales ---")
df_dist_locales = con.execute(f"""
    SELECT DISTINCT UPPER(TRIM(desc_distrito_local)) AS distrito
    FROM read_json_auto('{ficheros["locales"]}')
    WHERE desc_distrito_local IS NOT NULL
    ORDER BY distrito
""").fetchdf()
display(df_dist_locales)

print("\n--- Distritos en Airbnb (neighbourhood_group_cleansed) ---")
df_dist_listings = con.execute(f"""
    SELECT DISTINCT UPPER(TRIM(neighbourhood_group_cleansed)) AS distrito
    FROM read_json_auto('{ficheros["listings"]}')
    WHERE neighbourhood_group_cleansed IS NOT NULL
    ORDER BY distrito
""").fetchdf()
display(df_dist_listings)

## 6. Resumen de hallazgos

| Aspecto | Resultado |
|---------|----------|
| **Volumen total** | ~154,000 registros en datasets de Madrid + 16,313 en Airbnb |
| **Clave de integración principal** | `id_local` (locales ↔ actividades ↔ licencias ↔ terrazas) |
| **Relación locales ↔ Airbnb** | No hay FK directa; se relacionan por **distrito** y **coordenadas geográficas** |
| **Duplicados** | Solo en licencias (8 registros con misma PK, gestionados por fecha) |
| **Nulos relevantes** | `hora_apertura1`, `hora_cierre2` tienen alta tasa de nulos en locales |
| **Coordenadas locales** | En formato UTM EPSG:25830 → requieren conversión a WGS84 para MongoDB |
| **Coordenadas listings** | En GeoJSON pero con orden `[lat, lon]` incorrecto → requieren corrección a `[lon, lat]` |
| **Decisión de modelado** | Modelo **embebido** en MongoDB: actividades, licencias y terraza dentro de cada documento local |


> **Conclusión:** Los datos de Madrid (locales, actividades, licencias, terrazas) tienen una estructura relacional clara y coherente gracias al campo `id_local`. El dataset de Airbnb se integra de forma indirecta mediante el campo geográfico de distrito, lo cual es suficiente para el modelo propuesto en el Reto 2. La preparación completa del pipeline ETL se detalla en el notebook `01_data_preparation.ipynb`.